# Readme E-Codices

Dieses Jupyter Notebook bereitet Datenobjekte und Metadaten aus dem E-codices-Bestand vor für die digitale Langzeitarchivierung (DLZA) der ZHB Luzern, basierend auf der GOCFL implementierung von Jürgen Enge, https://github.com/je4/gocfl .



## Basiskonfiguration

Stand Juli 2023:
20 Titel (19 ZHB, 1 Korporation Luzern)
Datei mit Signaturen, DOI und Alma-ID im working directory, sowie auf https://stackfield.unilu.ch/run/val/bkkg/afchcf/bkaabheh ;

Angesichts der kleinen Menge wurde diese Datei von Hand zusammengeführt, da sich die Daten auf E-Codices (unifr) stark unterscheiden von den Daten in Alma (z.B. engl. und lateinische Titel/Autorennamen, keine Verlinkung von DOI/Alma-ID).

Für die E-Codices-Signaturen werden die DOI verwendet, welche sich aus den Original-Signaturen ableiten lassen. 
Bsp. doi:10.5076/e-codices-zhl-0034-4 / call number:  Msc.34.4

dlza signature: zhb_sosa_10_5076_e-codices-zhl-0034-4



### Config.py

Beispieldaten für ZHB E-codices. Anpassungen können in der config.py vorgenommen werden. 

    address = 'mailto:someone@internet.com'
    collection = 'ZHB E-Codices'
    collection_id = 'zhb_ecodices'
    ingest_workflow = 'W01'
    keywords = '[E-Codices, ZHB, Sondersammlung]'
    organisation = 'Zentral- und Hochschulbibliothek Luzern'
    organisation_id = 'zhb'
    sets = '[ecodices, zhb, sosa, lara]'
    signature = 'zhb_'
    
    
### Eingabedatei

Es wird nicht die OAI-Schnittstelle der UniFR verwendet, sondern eine Excel-Eingabedatei (e-codices.xlsx). Die Datei liegt im working directory.
Die Eingabedatei kann relativ leicht in Alma exportiert werden. Die E-codices sind in folgendem Set in der RZS gelistet: 
Set name: e-codices_dlza_heka - Itemized 
Die Export-Datei wurde leicht überarbeitet. Nicht benötigte Spalten wurden gelöscht, einige Daten auf mehrere Spalten aufgeteilt (z.B. Creator, Date), einige Daten händisch ergänzt (z.B. Dateipfade). Da die Sammlung überschaubar ist, hält sich der zeitliche Aufwand dafür in Grenzen.


### Export

Für jeden einzelnen record wird eine info.json-Datei erstellt im Format signature.json im directory info.
Das ganze Set wird am Ende noch als json- und Excel-Datei exportiert ins directory fulldump.

### Marcxml aus Alma (SRU)

Mit der alma_id werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert im directory metadata unter signature.xml


### Datenobjekte abholen
Die E-Codices-Objekte liegen auf G:\ZHB-Sosa_Digital\digital unter folgenden Pfaden:

Msc\ecod_Msc...
P\ecod_P...
Romero (Signatur)\ecod_Romero...
S\ecod_...

Der aktuelle Dateipfad ist in der Excel-Datei in Spalte "filepath" abgelegt und wird auch in die Infojson unter 'additional' abgelegt. 

### TODO create Befehle erstellen für gocfl 

Die gocfl create Befehle für die E-Codices-Sammlung (alle Objekte) werden gemäss https://github.com/je4/gocfl/blob/main/docs/create.md erstellt mit der Signature. 

Muster:
gocfl create P:/temp/archiv P:/temp/testdata/zhb_erara_86774/object metadata:p:/temp/testdata/zhb_erara_86774/metadata --config p:/temp/config/gocfl.toml -i "zhb_erara_86774" --ext-NNNN-metafile-source p:/temp/testdata/zhb_erara_86774.json 



In [1]:
import json
import config
import pandas as pd
import requests

# Read the Excel file into a pandas DataFrame
input_file = "e-codices.xlsx"
df = pd.read_excel(input_file)

# Convert the DataFrame to a list of dictionaries

completeSet = []

for _, row in df.iterrows():
    
    infoSet = {
        'additional': row['filepath'].replace('\\','/'),
        'address': config.address, 
        'collection': config.collection,
        'collection_id': config.collection_id ,
        'created': str(row['date']), 
        'identifiers': [str(row['alma_id']), row['doi'], row['han_number'], row['call number']],
        'ingest_workflow': config.ingest_workflow, 
        'keywords': config.keywords, 
        'last_changed': config.last_changed,
        'organisation' : config.organisation,
        'organisation_id' : config.organisation_id,
        'references' : [row['url']],
        'signature': config.signature+row['doi'].replace('.','_').replace('/','_'),
        'sets' : config.sets,
        'title' : row['title'],
        'user' : row['creator']      
    }
    
    signature = infoSet['signature']
    #print(signature)
    completeSet.append(infoSet)
    
    # Write the infoSet to a JSON file
    info_json = json.dumps(infoSet, indent=4, ensure_ascii=False)
    
    infofile = f"info/{signature}.json"
    with open(infofile, "w") as outfile:
        outfile.write(info_json)
        print(f"info.json saved as {infofile}")
    
    # get metadata from Alma OAI as MARCXML
    alma_id = str(row['alma_id'])   
    sru_url = "https://slsp-rzs.alma.exlibrisgroup.com/view/sru/41SLSP_RZS"
    query = f"{sru_url}?version=1.2&operation=searchRetrieve&recordSchema=marcxml&query=rec.id={alma_id}"
    response = requests.get(query)
    if response.status_code != 200:
        raise Exception(f"SRU request failed with status code {response.status_code}")

    # Save the response content (MARCXML) to a file
    metafile = f"metadata/{signature}.xml"
    with open(metafile, 'wb') as file:
        file.write(response.content)
        print(f"Record with ID {alma_id} saved as {metafile}")    


# Writing completeSet as json file

fulldump = json.dumps(completeSet, indent=4, ensure_ascii=False)
fulljsonfile = "fulldump/ecodices_complete_set.json"
with open(fulljsonfile, "w") as outfile:
    outfile.write(fulldump)
    print(f"---\nfulldump written to {fulljsonfile}")
    
# Writing completeSet as Excel file
fullexcelfile = "fulldump/ecodices_complete_set.xlsx"
df_json = pd.read_json(fulljsonfile)
df_json.to_excel(fullexcelfile)
print(f"Saved fulldump in excel file as {fullexcelfile}")


info.json saved as info/zhb_10_5076_e-codices-zhl-0034-4.json
Record with ID 9914249443205505 saved as metadata/zhb_10_5076_e-codices-zhl-0034-4.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0040.json
Record with ID 9914249441205505 saved as metadata/zhb_10_5076_e-codices-zhl-0040.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0042.json
Record with ID 9914249440305505 saved as metadata/zhb_10_5076_e-codices-zhl-0042.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0045.json
Record with ID 9914249439505505 saved as metadata/zhb_10_5076_e-codices-zhl-0045.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0043.json
Record with ID 9914249439005505 saved as metadata/zhb_10_5076_e-codices-zhl-0043.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0044.json
Record with ID 9914249437805505 saved as metadata/zhb_10_5076_e-codices-zhl-0044.xml
info.json saved as info/zhb_10_5076_e-codices-zhl-0041.json
Record with ID 9914249437605505 saved as metadata/zhb_10_5076_e-co